# 用反思改進 SQL 生成

## 1. 引言
### 1.1. 範例概覽

在本範例中，你將探索反思模式如何改進一個將自然語言問題轉換為 SQL 資料庫查詢的智慧體工作流。你將看到智慧體如何發現自身輸出中的問題、進行修正，並在給出最終答案前提升響應質量。

### 🎯 1.2 學習目標

你將練習應用反思模式，以提升智慧體工作流編寫 SQL 查詢的能力。為此，你將把一個反思步驟編碼到工作流中，使得智慧體能夠：

* 審查自身的中間結果（如 SQL 草稿或工具輸出）。
* 識別錯誤與缺漏。
* 檢查其響應與工具使用。
* 在提交最終答案前對輸出進行最佳化。

## 2. 環境設定：初始化環境與客戶端

在本步驟中，你將準備好工作空間以便立即開始編碼。你將：

1. **匯入核心 Python 庫**

   * `json`：處理結構化資料。
   * `pandas`：處理表格資料。
   * `dotenv`：載入環境變數（如 API 金鑰）。

2. **載入環境變數**
   以確保你的工作空間已正確配置所需的金鑰與設定。

3. **匯入 `utils.py` 模組**
   其中包含用於格式化輸出和支援後續工作流步驟的輔助函式。

**注意：** 如需檢視 `utils.py` 的內容，可在頂部選單選擇 **File > Open**。


In [1]:
import json
import utils
import pandas as pd
from dotenv import load_dotenv

_ = load_dotenv()

### 2.1 開始使用 AISuite

本專案的範例使用名為 aisuite 的包（[aisuite 倉庫](https://github.com/andrewyng/aisuite)），它讓你能夠方便地呼叫由不同提供商託管的 LLM。工具使用（Tool Use）的範例會進一步介紹 aisuite。現在初始化 `aisuite client`。該客戶端為你提供統一的方式連線並與不同 LLM 互動——無需擔心每個模型有各自的初始化方式。

In [2]:
import aisuite as ai

client = ai.Client()

### 2.2. 設定資料庫

在此步驟中，你將建立一個名為 **`products.db`** 的本地 SQLite 資料庫。
資料庫將自動填充隨機生成的產品資料。

你將在後續步驟中使用這些資料練習與測試 SQL 查詢。

In [3]:
utils.create_transactions_db()

已创建 SQLite 数据库 'products.db'，包含一个事件溯源的 'transactions' 表。


你可以執行下方單元以檢視資料表的架構（schema）。

In [4]:
utils.print_html(utils.get_schema('products.db'))

在該表中，每一行都代表一個**事件**（插入、補貨、銷售或價格更新）。關於庫存水平、銷售額或價格趨勢等分析都可由這些事件聚合得到。

<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#eff6ff; border-radius:6px; padding:12px 14px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">  
<strong>🔎 架構概覽：</strong><br><br>  
• <code>id</code> → 事件的唯一 ID（自增）。<br>  
• <code>product_id</code>、<code>product_name</code>、<code>brand</code>、<code>category</code>、<code>color</code> → 用於標識產品。<br>  
• <code>action</code> → 事件型別（<em>insert</em>、<em>restock</em>、<em>sale</em>、<em>price_update</em>）。<br>  
• <code>qty_delta</code> → 庫存變化（插入/補貨為 +，銷售為 –，價格更新為 0）。<br>  
• <code>unit_price</code> → 當時價格（補貨為 NULL）。<br>  
• <code>notes</code> → 事件的可選描述。<br>  
• <code>ts</code> → 記錄事件時的時間戳。<br>  
</div>  

> 基於該架構，你可以透過聚合事件歷史隨時重構**當前狀態**（庫存、最新價格、總銷售額）。

## 3. 構建 SQL 生成器

### 3.1. 使用 LLM 查詢資料庫

在本步驟中，你將使用一個函式把自然語言問題轉換為 SQL 查詢。

你將把問題與資料庫架構作為輸入提供給模型；隨後 LLM 會生成用於回答你問題的 SQL 查詢。

透過這種方式，**你**可以專注於提出問題，而由模型負責撰寫查詢語句。


In [5]:
def generate_sql(question: str, schema: str, model: str) -> str:
    prompt = f"""
    你是一名 SQL 助理。根據給定的資料庫架構與使用者問題，編寫適用於 SQLite 的 SQL 查詢。

    架構：
    {schema}

    使用者問題：
    {question}

    僅返回 SQL 語句。
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

執行下方單元，檢視 **`generate_sql`** 如何基於自然語言問題為 `transactions` 表生成 **第一版（V1）** SQL 查詢。
只需提供**問題**、**架構（schema）**與**模型名**，函式將返回初始查詢草稿。

In [6]:
# 示例：使用 generate_sql

# 我們將架構以字串形式提供
schema = """
表名: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

# 我們用自然語言提出關於資料的問題
question = "哪種顏色的產品總銷售額最高？"

utils.print_html(question, title="使用者問題")

# 使用指定模型生成 SQL 查詢
sql_V1 = generate_sql(question, schema, model="openai:gpt-4.1")

# 展示生成的 SQL 查詢
utils.print_html(sql_V1, title="SQL 查詢 V1")


#### 3.1.1. 查詢執行

現在你將執行**第一版（V1）SQL 查詢**並檢查其結果。此步驟很重要，因為它能幫你驗證 LLM 生成的查詢是否確實從資料庫檢索到了你期望的資訊。  

- **`utils.execute_sql(...)`**：針對 `products.db` 執行生成的 V1 查詢，並以 pandas DataFrame 返回輸出。使用 DataFrame 便於檢查、分析，並將結果傳遞到後續步驟。  

- **`utils.print_html(...)`**：將 DataFrame 渲染為整潔的 HTML 表格，使原始輸出更易讀，便於快速判斷查詢結果是否契合使用者問題。  


In [7]:
# 針對 products.db 執行生成的 SQL 查詢（sql_V1）。
# 結果將以 pandas DataFrame 返回。
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# 在筆記本中把 DataFrame 渲染為 HTML 表格。
# 這樣可使查詢輸出更易於閱讀與理解。
utils.print_html(df_sql_V1, title="SQL 查詢 V1 的輸出 - ❌ 未能完整回答問題")


color,total_sales
blue,-190571.46


<div style="border:1px solid #ef4444; border-left:6px solid #dc2626; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#7f1d1d; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

❌ <strong>V1 輸出問題：</strong> 查詢結果未能完整回答問題，因為總銷售額為<em>負值</em>（`-190571.46`）。  
<br><br>
在該資料集中，<code>qty_delta</code> 在**銷售**時記錄為**負數**（庫存減少），而在**退貨或補貨**時記錄為**正數**（庫存增加）。查詢使用了 <code>SUM(qty_delta)</code>，因此在跨交易求和時，負數佔主導導致總和為負。  
<br>
這意味著該 SQL 在技術上是有效的，但在語義上不正確——總銷售額不應以負值表示。  
<br>
➡️ 該問題<strong>進一步表明反思的必要性</strong>：模型需要最佳化其邏輯（例如在計算銷售額時將 <code>qty_delta</code> 乘以 <code>-1</code>，或使用 <code>ABS()</code>），使最終查詢能夠反映真實的銷售總額。  

</div>


### 3.2. 透過反思改進 SQL 查詢

在本節中，你將學習如何透過**反思**來改進 SQL 查詢。

首先，LLM 可僅根據問題與架構審查**SQL 文字**，在需要時給出改進建議。
隨後，LLM 還可考慮**實際查詢執行結果**，以捕捉負值總計、缺失篩選或分組錯誤等細微問題。

綜合上述方法，反思可以讓你的 SQL 工作流更**可靠且準確**——先在“紙面上”檢查邏輯，再用真實資料進行驗證。

#### 3.2.1. 初次嘗試：改進 SQL 查詢

在此函式中，你讓 LLM 根據原始問題與架構（如 3.1 節定義的架構）**審查**一條 SQL 查詢。模型會反思該查詢是否完整回答了問題；若沒有，則提出改進版本。

* **輸入**：

  * 使用者的**問題**
  * **原始 SQL 查詢**
  * **表架構（schema）**

* **輸出**：

  * **feedback** → 簡短評估（例如：“有效但缺少日期篩選”）
  * **refined_sql** → 最終 SQL（若原查詢正確則不變，若需改進則更新）

此函式**不執行 SQL**。它僅檢查查詢並在邏輯與意圖不完全匹配時給出最佳化建議。

In [8]:
def refine_sql(
    question: str,
    sql_query: str,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    反思查詢的“展示輸出”是否回答了使用者問題，
    如有需要，提出改進版 SQL。
    返回 (feedback, refined_sql)。
    """
    prompt = f"""
你是一位 SQL 審查與最佳化專家。

使用者問題：
{question}

原始 SQL：
{sql_query}

表架構：
{schema}

步驟 1：簡要評估 SQL 輸出是否完整回答使用者問題。
步驟 2：若需要改進，請提供適用於 SQLite 的最佳化版 SQL 查詢。
若原始 SQL 已正確，請保持不變返回。

嚴格返回僅包含以下兩個欄位的 JSON：
{{
  "feedback": "<1-3 句解釋問題或確認正確性>",
  "refined_sql": "<final SQL to run>"
}}
"""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        # 若模型未返回有效 JSON 的回退處理
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql


執行下方單元以生成**改進後的 SQL 查詢（V2）**。該步驟將：

* 顯示在 3.1 節針對問題“哪種顏色的產品總銷售額最高？”生成的**初始 SQL 查詢（V1）**
* 展示模型的**反饋**及其**改進版 SQL（V2）**
* 針對資料庫執行**原始 SQL（V1）**並呈現其**實際輸出**，幫助你瞭解為何需要改進。

In [9]:
# 示例：改進生成的 SQL（V1 → V2）

feedback, sql_V2 = refine_sql(
    question=question,
    sql_query=sql_V1,   # <- 來自 generate_sql()（V1）
    schema=schema, # <- 複用 3.1 節的架構
    model="openai:gpt-4.1"
)

# 展示原始問題
utils.print_html(question, title="使用者問題")

# --- V1 ---
utils.print_html(sql_V1, title="生成的 SQL 查詢（V1）")

# 執行並展示 V1 輸出
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')
utils.print_html(df_sql_V1, title="SQL 查詢 V1 的輸出 - ❌ 未能完整回答問題")

# --- 反饋 + V2 ---
utils.print_html(feedback, title="對 V1 的反饋")
utils.print_html(sql_V2, title="改進後的 SQL 查詢（V2）")

# 執行並展示 V2 輸出
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL 查詢 V2 的輸出 - ❌ 未能完整回答問題")

color,total_sales
blue,-190571.46


color,total_sales
blue,-190571.46


<div style="border:1px solid #fecaca; border-left:6px solid #dc2626; background:#fee2e2; border-radius:6px; padding:12px 14px; color:#7f1d1d; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

❌ **如你所見…**  
儘管生成的 SQL 看起來是正確的且反饋也予以確認，但**實際的查詢結果**顯示 `total_sales` 為負值。  

如前所述，這發生在 SQL 直接計算 `qty_delta * unit_price` 而未考慮銷售事件會記錄**負數數量**（`qty_delta < 0`）的情況下。正負號的反轉是一個細微的語義問題，單靠審查查詢文字並不總能發現。  

👉 因此智慧體還必須對**執行輸出**進行反思——以捕捉諸如符號錯誤、缺失的篩選或錯誤的聚合等問題。來自查詢結果的外部反饋使改進過程紮根於真實資料，而不僅僅是 SQL 的結構。  

</div>

#### 3.2.2. 最終方法：結合外部反饋改進 SQL 查詢

不同於上一步僅**審查 SQL 文字**，現在你將提供**實際的查詢執行結果**作為外部反饋。  
這些反饋來自在資料庫上執行 SQL 查詢因此 LLM 能使用真實輸出來評估查詢是否真正回答了問題。

在此步驟中，你將獲得：

* 基於**真實執行輸出**的簡短評估  
* 具體的改進建議（例如缺失篩選、分組問題、符號錯誤）  
* 更貼合原始問題的改進版 SQL 語句  

In [10]:
def refine_sql_external_feedback(
    question: str,
    sql_query: str,
    df_feedback: pd.DataFrame,
    schema: str,
    model: str,
) -> tuple[str, str]:
    """
    評估 SQL 結果是否回答使用者問題；如有必要，提出改進版查詢。
    返回 (feedback, refined_sql)。
    """
    prompt = f"""
    你是一位 SQL 審查與最佳化專家。

    使用者問題：
    {question}

    原始 SQL：
    {sql_query}

    SQL 輸出：
    {df_feedback.to_markdown(index=False)}

    表架構：
    {schema}

    步驟 1：簡要評估該 SQL 輸出是否回答了使用者問題。
    步驟 2：若可改進，請提供最佳化後的 SQL 查詢。
    若原始 SQL 已正確，請保持不變返回。

    請嚴格返回僅包含以下兩個欄位的 JSON 物件：
    - "feedback": 簡短評估與建議
    - "refined_sql": 需要執行的最終 SQL
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
    )

    
    content = response.choices[0].message.content
    try:
        obj = json.loads(content)
        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", sql_query)).strip()
        if not refined_sql:
            refined_sql = sql_query
    except Exception:
        # 若模型未返回有效 JSON 的回退處理：
        # 使用原始內容作為反饋並保留原始 SQL
        feedback = content.strip()
        refined_sql = sql_query

    return feedback, refined_sql

執行下方單元，觀察如何利用查詢結果的**外部反饋**來改進 SQL。

在此示例中你將：

* 展示基於你的問題生成的**原始 SQL 查詢（V1）**。
* 展示 **V1 的輸出**，說明為何初版未能完整回答問題。
* 基於該輸出提供 LLM 的**反饋**。
* 給出能夠解決問題的**改進版 SQL（V2）**。
* 執行 **V2** 並展示其輸出，確認它已 ✅ 完整回答問題。

In [11]:
# 示例：結合外部反饋改進 SQL（V1 → V2）

# 執行原始 SQL（V1）
df_sql_V1 = utils.execute_sql(sql_V1, db_path='products.db')

# 使用外部反饋進行評估與改進
feedback, sql_V2 = refine_sql_external_feedback(
    question=question,
    sql_query=sql_V1,   # V1 查詢
    df_feedback=df_sql_V1,    # V1 的輸出
    schema=schema,
    model="openai:gpt-4.1"
)

# --- V1 ---
utils.print_html(question, title="使用者問題")
utils.print_html(sql_V1, title="生成的 SQL 查詢（V1）")
utils.print_html(df_sql_V1, title="SQL 查詢 V1 的輸出 - ❌ 未能完整回答問題")

# --- 反饋與 V2 ---
utils.print_html(feedback, title="對 V1 的反饋")
utils.print_html(sql_V2, title="改進後的 SQL 查詢（V2）")

# 執行並展示 V2 結果
df_sql_V2 = utils.execute_sql(sql_V2, db_path='products.db')
utils.print_html(df_sql_V2, title="SQL 查詢 V2（結合外部反饋）的輸出 - ✅ 已完整回答問題")


color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


<div style="border:1px solid #bbf7d0; border-left:6px solid #22c55e; background:#dcfce7; border-radius:6px; padding:12px 14px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

✅ **成功！**  
透過改進後的 SQL，負值 `qty_delta` 問題已用 `ABS(qty_delta)`（或對銷售等價地使用 `-qty_delta`）修正。  

現在輸出為**正且更有意義**，顯示了總銷售額最高的正確顏色。這體現了結合**反思 + 外部反饋**的重要性：  
- 僅看 SQL 文字似乎沒有問題。  
- 實際執行輸出揭示了符號錯誤。  
- 改進後的查詢修正了邏輯，得到正確答案。  

</div>


### 3.3. 整體串聯 — 構建資料庫查詢工作流

在此步驟中，**你**將使用一個函式，將建立、執行與改進 SQL 查詢的整個流程自動化。

該工作流包括以下關鍵步驟：

1. **提取資料庫架構（schema）**
2. 從自然語言問題**生成初始（V1）SQL 查詢**
3. **結合執行反饋反思 V1** —— 審閱實際查詢結果並在需要時改進 SQL
4. **執行改進後的（V2）SQL 查詢**，確保它完整回答你的問題

最終，**你**將看到：

* 初始與改進後的查詢
* 各自的輸出
* LLM 提供的解釋改進的反饋

這讓**你**能夠更順暢、準確且全自動地處理 SQL 查詢。

In [ ]:
def run_sql_workflow(
    db_path: str,
    question: str,
    model_generation: str = "openai:gpt-4.1",
    model_evaluation: str = "openai:gpt-4.1",
):
    """
    端到端工作流：用於生成、執行、評估並改進 SQL 查詢。

    步驟：
      1) 提取資料庫架構
      2) 生成 SQL（V1）
      3) 執行 V1 → 展示輸出
      4) 結合執行反饋反思 V1 → 提出改進版 SQL（V2）
      5) 執行 V2 → 展示最終答案
    """

    # 1) 結構
    schema = utils.get_schema(db_path)
    utils.print_html(
        schema,
        title="📘 第 1 步 — 提取資料庫架構"
    )

    # 2) 生成 SQL（V1）
    sql_v1 = generate_sql(question, schema, model_generation)
    utils.print_html(
        sql_v1,
        title="🧠 第 2 步 — 生成 SQL（V1）"
    )

    # 3) 執行 V1 
    df_v1 = utils.execute_sql(sql_v1, db_path)
    utils.print_html(
        df_v1,
        title="🧪 第 3 步 — 執行 V1（SQL 輸出）"
    )

    # 4) 結合執行反饋反思 V1 → 提出改進版 SQL（V2）
    feedback, sql_v2 = refine_sql_external_feedback(
        question=question,
        sql_query=sql_v1,
        df_feedback=df_v1,          # 外部：V1 的執行結果
        schema=schema,
        model=model_evaluation,
    )
    utils.print_html(
        feedback,
        title="🧭 第 4 步 — 反思 V1（反饋）"
    )
    utils.print_html(
        sql_v2,
        title="🔁 第 4 步 — 改進後的 SQL（V2）"
    )

    # 5) 執行 V2 → 展示最終答案
    df_v2 = utils.execute_sql(sql_v2, db_path)
    utils.print_html(
        df_v2,
        title="✅ 第 5 步 — 執行 V2（最終答案）"
    )


### 3.4. 執行 SQL 工作流

現在輪到**你**來執行完整的 SQL 處理流水線。你可以嘗試以下 OpenAI 模型的不同組合，它們的能力與效能各不相同：

* `openai:gpt-4o`
* `openai:gpt-4.1`
* `openai:gpt-4.1-mini`
* `openai:gpt-3.5-turbo`

💡 在此工作流中，`openai:gpt-4.1` 往往在自我反思任務上表現最佳。

**重要說明：** 由於大語言模型（LLM）具有隨機性，每次執行可能會返回略有不同的結果。
建議你嘗試不同模型與組合，以找到最適合**你**的設定。

In [13]:
run_sql_workflow(
    "products.db", 
    "哪種顏色的產品總銷售額最高？",
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1"
)

color,total_sales
blue,-190571.46


color,total_sales
white,358315.09


## 4. 總結要點

完成本範例後，**你**學會了：

* 使用 LLM 將自然語言問題轉換為 SQL 查詢  
* 應用**反思模式**（包含/不包含外部反饋）改進生成的 SQL  
* 自動化完整的 SQL 工作流：從架構提取到查詢執行與改進  
* 嘗試不同的 LLM 模型，以比較效能與準確性  

關鍵洞察在於：反思讓**你的**智慧體更可靠。智慧體不再停留在第一次嘗試，而是能夠審查、改進，並給出更貼合**你的**意圖的結果。


<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>恭喜！</strong>  

你已完成**具備智慧體特性的 SQL 工作流（V1 → V2）**的構建。  

在此過程中，**你**練習瞭如何將反思、執行與驗證組合成一個可靠的管線。
你也見識了**外部反饋**的重要性：有時僅看 SQL 文字似乎正確，但實際執行輸出會暴露隱藏問題（如負數總計、缺失篩選或分組錯誤）。  

掌握這些技能後，**你**已準備好設計自己的**智慧體管線**——即：  
* 生成初始查詢（V1）  
* 結合執行反饋進行反思與改進（V2）  
* 交付更安全、更透明且更值得信賴的結果 🌟  

</div>
